# Group Project – Full Analysis Pipeline
**Authors:** Luo, Bassant, Osjah, Mickey, Alexia

This notebook consolidates all code across the project into a single, reproducible pipeline. 
Each section corresponds to a distinct analytical stage. 

**Pipeline stages:**
1. Data Cleaning (MapMF)
2. V-Dem Backsliding Index Construction
3. Baseline Bias
4. Sentiment Scoring (RoBERTa)
5. Dataset Merge (MapMF + V-Dem)
6. Baseline Model Comparison
7. Bias Audit & Category-Specific Correction
8. Exploratory Data Analysis
9. Multilevel Regression
10. Classification Reframe
11. Visualisations


In [ ]:
# Initial Data Files
MAPMF_RAW = "mapmf_alerts_cleaned.csv"
V_DEM = "V-Dem-CY-Full+Others-v16.csv"

## 1. Data Cleaning (MapMF)

Cleans the raw MapMF dataset. Steps:
- Drops rows missing `gender` or `type_of_incident`
- Removes unusable gender categories (Unknown, Not applicable, etc.)
- Splits pipe-delimited multi-gender entries into separate rows
- Retains only Man, Woman, and Non-binary journalists
- Filters to verbal attacks only

**Output:** `MAPMF_CLEANED`

In [ ]:
import pandas as pd

# Load the raw MapMF dataset
df = pd.read_csv(MAPMF_RAW)

# Drop rows missing key variables
df = df.dropna(subset=["gender", "type_of_incident"])

# Standardize the gender column
df["gender"] = df["gender"].astype(str).str.strip()

# Remove unusable gender categories
unusable = ["Unknown gender", "Not applicable", "no data collected"]
for val in unusable:
    df = df[~df["gender"].str.contains(val, case=False, na=False)]

# Split pipe-delimited multi-gender entries into separate rows
# e.g. 'Man | Woman' becomes two rows, one per gender
df["gender"] = df["gender"].str.split("|")
df = df.explode("gender")
df["gender"] = df["gender"].str.strip()

# Keep only the three recognised gender categories
df = df[df["gender"].isin(["Man", "Woman", "Non-binary"])]

print("Gender distribution before verbal-attack filter:")
print(df["gender"].value_counts())

# Retain only verbal attack incidents
df = df[df["type_of_incident"].str.contains("Verbal attack", case=False, na=False)]

print(f"\nFinal rows after filtering: {len(df)}")
print(df["gender"].value_counts())

# Save cleaned dataset
df.to_csv("mapmf_gender_cleaned.csv", index=False)
print(f"\nCleaned dataset saved")


## 2. V-Dem Backsliding Index Construction

Builds a continuous democratic backsliding score for each country-year.

**Method:**
- Loads V-Dem global composite scores
- For each country, calculates year-on-year changes using only consecutive years
- A year is classified as backsliding when: (1) the single-year decline exceeds threshold tau, 
AND (2) the 3-year slope is negative, AND (3) the 3-year net change also exceeds tau
- `backslide_score` sums the magnitudes of short- and medium-term decline as a continuous severity measure

**Output:** `VDEM_RESULTS`, `VDEM_SUMMARY`

### 2a. V-Dem Composite Score Construction (alternative: weighted z-score method)


In [ ]:
# This cell constructs the composite democracy score from raw V-Dem indicators.

import pandas as pd

# Load the full V-Dem dataset (large file, may take a moment)
df_vdem_raw = pd.read_csv(V_DEM, low_memory=False)

# Select the 10 theoretically motivated indicators + identifiers
indicator_cols = [
    "v2x_freexp_altinf",  # Freedom of expression and alternative information (weight: 0.25)
    "v2x_jucon",          # Judicial constraints on the executive           (weight: 0.20)
    "v2xlg_legcon",       # Legislative constraints on the executive        (weight: 0.15)
    "v2csreprss",         # Civil society repression                        (weight: 0.10)
    "v2elintim",          # Electoral intimidation                          (weight: 0.08)
    "v2elirreg",          # Electoral irregularities                        (weight: 0.07)
    "v2exbribe",          # Executive bribery and corrupt exchanges         (weight: 0.05)
    "v2csprtcpt",         # Civil society participatory environment         (weight: 0.05)
    "v2elmulpar",         # Elections multiparty                            (weight: 0.03)
    "v2elfrfair",         # Election free and fair                          (weight: 0.02)
]

df_sub = df_vdem_raw[["country_name", "year"] + indicator_cols].copy()

# Standardize each indicator to z-scores
for col in indicator_cols:
    df_sub[f"{col}_z"] = (df_sub[col] - df_sub[col].mean()) / df_sub[col].std()

# Build weighted composite score (higher = stronger democratic institutions)
weights = [0.25, 0.20, 0.15, 0.10, 0.08, 0.07, 0.05, 0.05, 0.03, 0.02]
df_sub["composite_score"] = sum(
    w * df_sub[f"{col}_z"] for w, col in zip(weights, indicator_cols)
)

# Calculate 1-year and 4-year backsliding deltas per country
df_sub = df_sub.sort_values(["country_name", "year"]).copy()
df_sub["delta_1y"] = df_sub.groupby("country_name")["composite_score"].diff(1)
df_sub["delta_4y"] = df_sub.groupby("country_name")["composite_score"].diff(4)

print("Composite score constructed. Shape:", df_sub.shape)
print(df_sub[["country_name", "year", "composite_score", "delta_1y"]].head())

df_sub.to_csv("V-dem-composite.csv", index=False)
print(f"\nComposite dataset saved")



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("V-dem-composite.csv")

if "country_name" not in df.columns:
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

df.columns = df.columns.astype(str).str.strip()

required_cols = ["country_name", "year", "composite_score"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}\nAvailable columns: {df.columns.tolist()}")

df["country_name"] = df["country_name"].astype(str).str.strip()
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["composite_score"] = pd.to_numeric(df["composite_score"], errors="coerce")

df = df.dropna(subset=["country_name", "year", "composite_score"]).copy()
df["year"] = df["year"].astype(int)

df = df.sort_values(["country_name", "year"]).reset_index(drop=True)

print("Columns loaded correctly:")
print(df.columns.tolist())
print(df.head())

def classify_backsliding(group, k=3, min_abs_drop=0.05):
    group = group.sort_values("year").copy()

    if "country_name" not in group.columns:
        group = group.reset_index()

    contiguous = (group["year"] - group["year"].shift(1) == 1)

    group["delta_1yr"] = np.where(
        contiguous,
        group["composite_score"] - group["composite_score"].shift(1),
        np.nan
    )

    valid_deltas = group["delta_1yr"].dropna()
    if len(valid_deltas) > 4:
        tau = max(min_abs_drop, valid_deltas.std())
    else:
        tau = min_abs_drop

    group["tau"] = tau
    group["decline_year"] = (group["delta_1yr"] < -tau).astype(int)

    slopes = []
    net_changes = []
    decline_counts = []

    for i in range(len(group)):
        if i < k - 1:
            slopes.append(np.nan)
            net_changes.append(np.nan)
            decline_counts.append(np.nan)
            continue

        window = group.iloc[i-k+1:i+1]
        years = window["year"].values
        values = window["composite_score"].values

        if np.all(np.diff(years) == 1):
            slope = np.polyfit(years, values, 1)[0]
            net_change = values[-1] - values[0]
            count_declines = np.sum(np.diff(values) < -tau)
        else:
            slope = np.nan
            net_change = np.nan
            count_declines = np.nan

        slopes.append(slope)
        net_changes.append(net_change)
        decline_counts.append(count_declines)

    group[f"slope_{k}yr"] = slopes
    group[f"net_change_{k}yr"] = net_changes
    group[f"decline_count_{k}yr"] = decline_counts

    group["backsliding_year"] = (
        (group["delta_1yr"] < -tau) &
        (group[f"slope_{k}yr"] < 0) &
        (group[f"net_change_{k}yr"] < -tau) &
        (group[f"decline_count_{k}yr"] >= 2)
    ).astype(int)

    group["stable_deterioration"] = (
        (group["delta_1yr"] >= -tau) &
        (group[f"slope_{k}yr"] < 0) &
        (group[f"net_change_{k}yr"] < -tau) &
        (group[f"decline_count_{k}yr"] >= 2)
    ).astype(int)

    group["backslide_score"] = (
        np.maximum(0, -group["delta_1yr"].fillna(0)) +
        np.maximum(0, -group[f"net_change_{k}yr"].fillna(0)) +
        np.maximum(0, -(group[f"slope_{k}yr"].fillna(0) * k))
    )

    return group


df_result = (
    df.groupby("country_name", as_index=False, group_keys=False)
      .apply(classify_backsliding)
      .reset_index(drop=True)
)

# Safety check: restore Country if it became index
if "country_name" not in df_result.columns:
    df_result = df_result.reset_index()
    if "country_name" not in df_result.columns:
        # rebuild from original df if needed
        df_result["country_name"] = df["country_name"].values[:len(df_result)]

print("\nColumns in df_result:")
print(df_result.columns.tolist())

preview_cols = [col for col in [
    "country_name", "year", "composite_score", "delta_1yr",
    "decline_year", "backsliding_year",
    "stable_deterioration", "backslide_score"
] if col in df_result.columns]

print("\nProcessed data preview:")
print(df_result[preview_cols].head(10))

summary = df_result.groupby("country_name", as_index=False).agg(
    deterioration_years=("decline_year", "sum"),
    backsliding_years=("backsliding_year", "sum"),
    stable_deterioration_years=("stable_deterioration", "sum"),
    avg_backslide_score=("backslide_score", "mean"),
    total_net_change=("composite_score", lambda x: x.iloc[-1] - x.iloc[0]),
    first_year=("year", "min"),
    last_year=("year", "max")
)

print("\nCountry summary:")
print(summary.head(10))


df_result.to_csv("vdem_backsliding_results.csv", index=False)
summary.to_csv("vdem_europe_summary.csv", index=False)

print("\nSaved files:")
print("- vdem_backsliding_results.csv")

country_name = "Hungary"  

country_df = df_result[df_result["country_name"] == country_name].sort_values("year").copy()

if len(country_df) == 0:
    print(f"\nNo data found for {country_name}")
else:
    plt.figure(figsize=(11, 6))

    plt.scatter(country_df["year"], country_df["composite_score"], label="Yearly value")
    plt.plot(country_df["year"], country_df["composite_score"], alpha=0.7)

    valid = country_df[["year", "composite_score"]].dropna()
    if len(valid) >= 2:
        z = np.polyfit(valid["year"], valid["composite_score"], 1)
        p = np.poly1d(z)
        plt.plot(valid["year"], p(valid["year"]), linestyle="--", label="Trendline")

    backslide = country_df[country_df["backsliding_year"] == 1]
    if len(backslide) > 0:
        plt.scatter(backslide["year"], backslide["composite_score"], s=90, label="Backsliding year")

    plt.title(f"Democratic trajectory over time: {country_name}")
    plt.xlabel("Year")
    plt.ylabel("Composite democracy score")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    

## 3. Baseline Bias

Model: SiEBERT (fine-tuned RoBERTa-Large model for sentiment analysis)
Since the model will be applied to verbal attacks that vary widely in syntactic structure, verb choice, and context, the baseline must be estimated across comparably varied neutral sentences to produce a stable, generalisable correction value. After reviewing the literature on gender bias measurement in NLP, 50 distinct sentence templates, organised into 5 structural categories, were chosen. 

1. Simple action templates           (Lu et al., 2020)
2. Identity / role templates         (Kiritchenko & Mohammad, 2018)
3. Event-based templates             (Kiritchenko & Mohammad, 2018)
4. Institutional / formal templates  (Lu et al., 2020)
5. Communication templates           (Lu et al., 2020)

Each template is professional, contains no adjectives and no emotional language, and varies only the pronoun (he / she) between paired sentences. This isolates model-level gender bias from content, following the counterfactual paired-sentence approach of Kiritchenko & Mohammad (2018).

Bias = Mean(Female Score - Male Score) across all 50 sentence pairs.

References:

Kiritchenko, S., & Mohammad, S. (2018, June 1). Examining Gender and Race Bias in Two Hundred Sentiment Analysis Systems. ACLWeb; Association for Computational Linguistics. https://doi.org/10.18653/v1/S18-2005

Lu, K., Mardziel, P., Wu, F., Amancharla, P., & Datta, A. (2020). Gender Bias in Neural Natural Language Processing. Logic, Language, and Security, 12300, 189–202. https://doi.org/10.1007/978-3-030-62077-6_14

In [ ]:

# packages
import torch
import pandas as pd
import numpy as np
from transformers import pipeline
# sentence templates
# [placeholder] = pronoun placeholder

templates = {
    "simple_action": [
        "[placeholder] filed the report.",
        "[placeholder] attended the briefing.",
        "[placeholder] submitted the document.",
        "[placeholder] completed the assignment.",
        "[placeholder] conducted the interview.",
        "[placeholder] reviewed the footage.",
        "[placeholder] published the findings.",
        "[placeholder] contacted the source.",
        "[placeholder] covered the event.",
        "[placeholder] presented the results.",
    ],

    "identity_role": [
        "[placeholder] is a journalist.",
        "[placeholder] is a reporter.",
        "[placeholder] is a correspondent.",
        "[placeholder] is an editor.",
        "[placeholder] works as a press officer.",
        "[placeholder] works as a news anchor.",
        "[placeholder] is a media professional.",
        "[placeholder] is a photographer.",
        "[placeholder] works as a broadcaster.",
        "[placeholder] is a contributor to the publication.",
    ],

    "event_based": [
        "[placeholder] attended the press conference.",
        "[placeholder] reported from the scene.",
        "[placeholder] covered the parliamentary session.",
        "[placeholder] appeared at the media briefing.",
        "[placeholder] was present at the hearing.",
        "[placeholder] reported on the court proceedings.",
        "[placeholder] attended the summit.",
        "[placeholder] covered the election.",
        "[placeholder] reported from the field.",
        "[placeholder] was at the scene of the incident.",
    ],

    "institutional": [
        "[placeholder] submitted the article to the editor.",
        "[placeholder] signed the press accreditation form.",
        "[placeholder] filed a freedom of information request.",
        "[placeholder] registered for the press event.",
        "[placeholder] applied for a press pass.",
        "[placeholder] submitted the investigation to the publication.",
        "[placeholder] requested a statement from the ministry.",
        "[placeholder] filed the story before the deadline.",
        "[placeholder] sent the report to the newsroom.",
        "[placeholder] submitted a complaint to the press council.",
    ],

    "communication": [
        "[placeholder] spoke to the spokesperson.",
        "[placeholder] contacted the press office.",
        "[placeholder] interviewed the official.",
        "[placeholder] asked the minister for comment.",
        "[placeholder] reached out to the organisation.",
        "[placeholder] corresponded with the source.",
        "[placeholder] informed the editor of the update.",
        "[placeholder] spoke at the press conference.",
        "[placeholder] consulted with the legal team.",
        "[placeholder] communicated the findings to the desk.",
    ],

}

# flatten / combine templates into single list
combined_templates = []
categories = []
for category, sentences in templates.items():
    for s in sentences:
        combined_templates.append(s)
        categories.append(category)

# stops before scoring if not all the templates are there
assert len(combined_templates) == 50, f"Expected 50 templates, got {len(combined_templates)}."
# he / she sentence pairs
he_sentences = [s.replace("[placeholder]", "He") for s in combined_templates]
she_sentences = [s.replace("[placeholder]", "She") for s in combined_templates]
# load SiEBERT model and tokenizer
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="siebert/sentiment-roberta-large-english",
    tokenizer="siebert/sentiment-roberta-large-english",
    truncation=True,
    max_length=512, # to prevent crashing, not relevant rn but good practice for longer sentences later if applicale
)

print("Model loaded.")
# model is a binary classifier (positive/negative) and only returns score for the label it's most confident about
# return result["score"] if result["label"] == "POSITIVE" else 1.0 - result["score"] keeps the score on the same scale, so we're not comparing positive score for one sentence to negative score for another
# function to return positive sentiment (0 to 1), lower more negative
def get_score(text: str) -> float:
    result = sentiment_pipeline(text)[0]
    return result["score"] if result["label"] == "POSITIVE" else 1.0 - result["score"]

# score sentences
he_scores = [get_score(s) for s in he_sentences]
she_scores = [get_score(s) for s in she_sentences]
# results table
results = pd.DataFrame({
    "category": categories,
    "template": combined_templates,
    "he_sentence": he_sentences,
    "she_sentence": she_sentences,
    "he_score": he_scores,
    "she_score": she_scores,
    "difference": [s - h for s, h in zip(she_scores, he_scores)],
})
# calculate bias
bias = results["difference"].mean()

print(f"Bias = {bias:.6f}") # bias might be very small, prints with 6 decimal places for precision
if bias > 0:
    print("The model scores she-sentences more positively than he-sentences. Without correction, verbal attacks on women would appear *less* severe.")
elif bias < 0:
    print("The model scores she-sentences more negatively than he-sentences. Without correction, verbal attacks on women would appear *more* severe.")
else:
    print("The model scores she-sentences and he-sentences equally. No bias detected in this baseline test.")
# extra breakdown by category
print(f"Breakdown by category:")
for category in templates.keys():
    subset = results[results["category"] == category]
    print(f"  {category:25s}: mean diff = {subset['difference'].mean():.6f}  "
          f"(std = {subset['difference'].std():.6f})")
# overall summary
print(f"Overall summary:")
print(f"  mean he-score:   {results['he_score'].mean():.6f}")
print(f"  mean she-score:  {results['she_score'].mean():.6f}")
print(f"  std of diffs:    {results['difference'].std():.6f}")
print(f"  min diff:        {results['difference'].min():.6f}")
print(f"  max diff:        {results['difference'].max():.6f}")
# extra save outputs to csv and create plain text file with just bias number for easy reference later
results.to_csv("baseline_bias_results.csv", index=False)
with open("baseline_bias_value", "w") as f:
    f.write(str(bias))

## 4. Sentiment Scoring (RoBERTa)

Scores each incident using Cardiff NLP's `twitter-roberta-base-sentiment-latest` model.

**Method:**
- Each incident is processed twice: once with original pronouns, once with pronouns swapped
- The model returns a weighted polarity score on a continuous scale (-1 to +1)
- Gendered intensity = (female version score - male version score) - global bias correction
- A positive gendered_intensity score indicates the female framing was scored as more hostile

**Note:** This cell requires a GPU or MPS (Apple Silicon) for reasonable runtime. 
CPU processing of 3,000+ incidents will be very slow.


In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "siebert/sentiment-roberta-large-english"
INPUT_FILE = "mapmf_gender_cleaned.csv"


BIAS = bias


print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, use_safetensors=True)


def get_sentiment_score(text):
    """
    Returns the probability of the POSITIVE class (index 1).
    """
    if pd.isna(text) or str(text).strip() == "": return 0.5
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Softmax to get probabilities (Scale 0 to 1)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # Return index 1: Positive probability
    return float(probs[0][1].item())

def targeted_victim_swap(text, primary_gender):
    """Swaps pronouns for the victim while protecting side characters."""
    if pd.isna(text) or str(text).strip() == "": return text
    doc = nlp(text)
    output = []
    woman_to_man = {"she": "he", "her": "him", "hers": "his", "herself": "himself"}
    man_to_woman = {"he": "she", "him": "her", "his": "her", "himself": "herself"}
    active_map = woman_to_man if primary_gender == "Woman" else man_to_woman if primary_gender == "Man" else {}
    
    for token in doc:
        word_low = token.text.lower()
        new_word = token.text
        if word_low in active_map:
            if primary_gender == "Woman" and word_low == "her":
                new_word = "his" if (token.pos_ in ["DET", "PRON"] and token.dep_ == "poss") else "him"
            else:
                new_word = active_map[word_low]
            if token.text.istitle(): new_word = new_word.capitalize()
        output.append(new_word + token.whitespace_)
    return "".join(output)



# Loading the full file
df = pd.read_csv(INPUT_FILE)

print(f"Processing {len(df)} rows...")

# Step 1: Create Parallel Corpus
df['parallel_content'] = df.apply(lambda r: targeted_victim_swap(r['content'], r['primary_gender']), axis=1)

# Step 2: Get Sentiment Scores
tqdm.pandas(desc="Scoring Original")
df['score_original'] = df['content'].progress_apply(get_sentiment_score)

tqdm.pandas(desc="Scoring Parallel")
df['score_parallel'] = df['parallel_content'].progress_apply(get_sentiment_score)

# Step 3: Intensity = (Sentiment Female – Sentiment Male) – Bias
def calculate_intensity(row):
    s_orig = row['score_original']
    s_para = row['score_parallel']
    
    if row['primary_gender'] == "Woman":
        # Original (F) - Parallel (M)
        return (s_orig - s_para) - BIAS
    elif row['primary_gender'] == "Man":
        # Parallel (F) - Original (M)
        return (s_para - s_orig) - BIAS
    else:
        return 0.0

df['gendered_intensity'] = df.apply(calculate_intensity, axis=1)

# Save the final data
df.to_csv("gendered_intensity.csv", index=False)
print(f"Final column 'gendered_intensity' created.")

## 5. Dataset Merge (MapMF + RoBERTa + V-Dem)

Merges the RoBERTa-scored MapMF incident data which includes `gendered_intensity` with V-Dem backsliding scores at the country-year level. 

In [ ]:


mapmf   = pd.read_csv("gendered_intensity.csv")
vdem    = pd.read_csv("vdem_backsliding_results.csv")
vdem_full = pd.read_csv("V-Dem-CY-Full+Others-v16.csv", low_memory=False)

# MapMF and V-Dem use different spellings for some countries.
# This map ensures they align correctly on the merge key.
country_mapping = {
    "Turkey":          "Türkiye",
    "Russia":          "Russian Federation",
    "Czech Republic":  "Czechia",
    "Slovak Republic": "Slovakia",
    "Bosnia":          "Bosnia and Herzegovina",
    "Macedonia":       "North Macedonia",
}

# Standardize country name columns
# (Note: vdem and vdem_full both use "country_name"/"year" as column names --
# not "Country"/"Year" -- matching the real V-Dem output headers.)
for df_obj, col in [(mapmf, "country"), (vdem, "country_name"), (vdem_full, "country_name")]:
    df_obj[col] = df_obj[col].astype(str).str.strip()

# Create harmonized merge keys
mapmf["country_for_merge"]     = mapmf["country"].replace(country_mapping)
vdem["country_for_merge"]      = vdem["country_name"].replace(country_mapping)
vdem_full["country_for_merge"] = vdem_full["country_name"].replace(country_mapping)


# 2025 and 2026 data collection was incomplete at time of analysis
mapmf["year"] = pd.to_numeric(mapmf["year"], errors="coerce")
mapmf = mapmf[~mapmf["year"].isin([2025, 2026])].copy()

# Standardize year types
vdem["year"] = pd.to_numeric(vdem["year"], errors="coerce")
vdem_full["year"] = pd.to_numeric(vdem_full["year"], errors="coerce")


merged = pd.merge(
    mapmf,
    vdem,
    on=["country_for_merge", "year"],
    how="left"
)

# add structural gender variables from full V-Dem 
structural_gender_vars = [
    "v2clacjstw",   # Women's property rights
    "v2csgender",   # Civil society women's participation
    "v2mefemjrn",   # Female journalists
    "v2cldiscw",    # Freedom of discussion for women
    "v2pepwrgen",   # Power distributed by gender
    "v2lgfemleg",   # Female legislators
    "v2cldmovew",   # Women's freedom of movement
    "v2x_gender",   # Women's political empowerment index (used as a predictor in Section 6)
]

vdem_gender = (
    vdem_full[["country_for_merge", "year"] + structural_gender_vars]
    .drop_duplicates(subset=["country_for_merge", "year"])
    .copy()
)

merged = pd.merge(
    merged,
    vdem_gender,
    on=["country_for_merge", "year"],
    how="left"
)

# Clean up redundant columns 
merged.drop(columns=[c for c in ["country_for_merge"] if c in merged.columns], inplace=True)

# Confirm RoBERTa sentiment scores and V-Dem backsliding scores
required_in_merge = ["gendered_intensity", "backslide_score"]
missing_in_merge = [c for c in required_in_merge if c not in merged.columns]
assert not missing_in_merge, (
    f"Merged dataset is missing {missing_in_merge} — check that mapmf was loaded "
    f"from ROBERTA_RESULTS (Section 4's output) and that VDEM_RESULTS includes backslide_score."
)
print(f"Confirmed: {required_in_merge} are both present in the merged dataset.")

# Check missingness in gender variables
print("Missing values in structural gender variables:")
print(merged[structural_gender_vars].isna().sum())

# Save merged dataset
merged.to_csv("final_merged_dataset.csv", index=False)
print(f"\nMerge complete. Shape: {merged.shape}")
print(f"Saved to: {"final_merged_dataset.csv"}")


## 6. Baseline Model Comparison


**Models tested:**
- Model 1: Linear Regression (simplest baseline, tests linear relationships)
- Model 2: Random Forest (captures non-linear relationships)
- Model 3: Gradient Boosting (step-by-step ensemble, sensitive to smaller patterns)
- Model 4: Multilevel Regression (accounts for country-level grouping structure)

All models use the same predictors and the same train/test split for a fair comparison. 
`backslide_score` is deliberately excluded here as it is reserved for the final model.

**Evaluation metrics:** RMSE, MAE, R² 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.formula.api as smf

# Load the merged dataset
df_model = pd.read_csv("final_merged_dataset.csv")

# Predictors used across all baseline models
# Note: backslide_score is excluded here; it is used only in the final model
feature_cols = [
    "primary_gender",  # Journalist gender (categorical)
    "country",         # Country of incident (categorical, captures fixed effects)
    "year",            # Year of incident (controls for time trends)
    "v2pepwrgen",      # Power distributed by gender
    "v2cldiscw",       # Freedom of discussion for women
    "v2mefemjrn",      # Female journalists indicator
    "v2x_gender",      # Gender equality index
]
target_col = "gendered_intensity"

# Drop rows with missing values in any required column
needed_cols = feature_cols + [target_col]
missing_cols = [c for c in needed_cols if c not in df_model.columns]
if missing_cols:
    print("Warning: missing columns:", missing_cols)

df_model = df_model.dropna(subset=needed_cols).copy()
df_model["year"] = pd.to_numeric(df_model["year"], errors="coerce")
df_model = df_model.dropna(subset=["year"])
df_model["year"] = df_model["year"].astype(int)

print(f"Rows used for modelling: {df_model.shape[0]}")
print(f"Years covered: {df_model['year'].min()} - {df_model['year'].max()}")

# Train / test split
X = df_model[feature_cols]
y = df_model[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing 
preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["primary_gender", "country"]),
    ("num", "passthrough", ["year", "v2pepwrgen", "v2cldiscw", "v2mefemjrn", "v2x_gender"])
])

# Model 1: Linear Regression 
model1 = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())])
model1.fit(X_train, y_train)
pred1 = model1.predict(X_test)

# Model 2: Random Forest 
model2 = Pipeline([("preprocessor", preprocessor), ("model", RandomForestRegressor(random_state=42))])
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)

# Model 3: Gradient Boosting
model3 = Pipeline([("preprocessor", preprocessor), ("model", GradientBoostingRegressor(random_state=42))])
model3.fit(X_train, y_train)
pred3 = model3.predict(X_test)

#  Model 4: Multilevel Regression 
# Country is used as the grouping structure (random intercepts per country)
train_data = X_train.copy(); train_data[target_col] = y_train
test_data  = X_test.copy();  test_data[target_col]  = y_test

model4 = smf.mixedlm(
    formula="gendered_intensity ~ C(primary_gender) + year + v2pepwrgen + v2cldiscw + v2mefemjrn + v2x_gender",
    data=train_data,
    groups=train_data["country"]
).fit(reml=False)
pred4 = model4.predict(test_data)

# Comparison table
results = pd.DataFrame({
    "Model": [
        "Model 1: Linear Regression",
        "Model 2: Random Forest",
        "Model 3: Gradient Boosting",
        "Model 4: Multilevel Regression"
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, pred1)),
        np.sqrt(mean_squared_error(y_test, pred2)),
        np.sqrt(mean_squared_error(y_test, pred3)),
        np.sqrt(mean_squared_error(y_test, pred4))
    ],
    "MAE": [
        mean_absolute_error(y_test, pred1),
        mean_absolute_error(y_test, pred2),
        mean_absolute_error(y_test, pred3),
        mean_absolute_error(y_test, pred4)
    ],
    "R2": [
        r2_score(y_test, pred1),
        r2_score(y_test, pred2),
        r2_score(y_test, pred3),
        r2_score(y_test, pred4)
    ]
}).sort_values("RMSE")

print("Model comparison (sorted by RMSE, lower is better):")
print(results.round(6).to_string(index=False))

print("\nTarget variable distribution:")
print(df_model["gendered_intensity"].describe().round(6))

print("\nCorrelation with year:")
print(df_model[["gendered_intensity", "year"]].corr().round(6))

print("\nMean gendered_intensity by gender:")
print(df_model.groupby("primary_gender")["gendered_intensity"].mean().round(6))


## 7. Bias Audit and Category-Specific Correction

Implements the two-stage bias correction described in the Bias Audit section of the report.

**Stage 1 (already applied in Section 4):** Global bias subtracted from all scores.

**Stage 2 (this section):** Category-specific correction.
Within each attack category, the residual gender gap (mean female score - mean male score) 
is subtracted from female-journalist rows. This removes systematic differences that vary 
across attack types and were not addressed by the global correction.

The comparison table shows why the global correction alone was insufficient: 
bias in death threats is ~20x larger than the global constant.

**Output:** `BIAS_CORRECTED`, plus three diagnostic CSV tables.

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


# Read the global bias value produced 
with open(bias) as f:
    GLOBAL_BIAS = float(f.read().strip())

# Map raw type_of_incident sub-labels to canonical category names
SUBLABEL_MAP = {
    "Sexual harassment":                  "Sexual harassment",
    "Harassment, insult, bullying":        "Harassment/insult/bullying",
    "Intimidation/threatening":            "Intimidation/threatening",
    "Intimidation/threatening unspecific": "Intimidation/threatening",  # merge variant
    "Discredit":                           "Discredit",
    "Death threat":                        "Death threat",
    "Defamation":                          "Defamation",
}

CATEGORY_ORDER = [
    "Death threat", "Sexual harassment", "Harassment/insult/bullying",
    "Discredit", "Intimidation/threatening", "Defamation"
]


df = pd.read_csv("final_merged_dataset.csv")

# Explode type_of_incident to get one row per sub-label 
# Incidents can have multiple sub-labels (e.g. 'Death threat | Defamation')
df["_types_list"] = df["type_of_incident"].str.split(r"\s*\|\s*")
df_exp = df.explode("_types_list").copy()
df_exp["_types_list"] = df_exp["_types_list"].str.strip()
df_exp["sub_label"]   = df_exp["_types_list"].map(SUBLABEL_MAP)
df_verbal = df_exp.dropna(subset=["sub_label"]).copy()

print(f"Rows after explode: {len(df_verbal):,}")
print(f"Categories: {sorted(df_verbal['sub_label'].unique())}\n")

# Compute category-specific bias 
# Bias = mean(gendered_intensity | Woman) - mean(gendered_intensity | Man) per category
mean_cg = (
    df_verbal
    .groupby(["sub_label", "gender"])["gendered_intensity"]
    .mean()
    .unstack("gender")
    .rename_axis(None, axis=1)
)
mean_cg["category_bias"] = mean_cg["Woman"] - mean_cg["Man"]
category_bias = mean_cg["category_bias"].to_dict()

print("Mean gendered_intensity per category per gender:")
print(mean_cg.round(6).to_string())

# Apply global correction 
df_verbal["gi_global_corrected"] = df_verbal["gendered_intensity"] - GLOBAL_BIAS

# Apply category-specific correction (female rows only)
def category_correction(row):
    """Subtracts the category-level bias from female-journalist rows."""
    bias = category_bias.get(row["sub_label"], 0.0)
    return bias if row["gender"] == "Woman" else 0.0

df_verbal["_cat_corr"] = df_verbal.apply(category_correction, axis=1)
df_verbal["gi_category_corrected"] = df_verbal["gendered_intensity"] - df_verbal["_cat_corr"]

# Residual bias analysis 
def residual_bias(column_name):
    """Returns the Woman-Man gap per category after a given correction."""
    return (
        df_verbal.groupby(["sub_label", "gender"])[column_name]
        .mean().unstack("gender")
        .assign(residual=lambda x: x["Woman"] - x["Man"])["residual"]
    )

res_original = residual_bias("gendered_intensity")
res_global   = residual_bias("gi_global_corrected")
res_category = residual_bias("gi_category_corrected")

# Comparison table 
comparison = pd.DataFrame({
    "Category bias (data)":  [category_bias[c] for c in CATEGORY_ORDER],
    "Global bias (constant)":[GLOBAL_BIAS] * len(CATEGORY_ORDER),
    "Difference":            [category_bias[c] - GLOBAL_BIAS for c in CATEGORY_ORDER],
}, index=CATEGORY_ORDER)

residual_df = pd.DataFrame({
    "No correction":      res_original.reindex(CATEGORY_ORDER),
    "Global correction":  res_global.reindex(CATEGORY_ORDER),
    "Category correction":res_category.reindex(CATEGORY_ORDER),
})

mar_none     = res_original.abs().mean()
mar_global   = res_global.abs().mean()
mar_category = res_category.abs().mean()

bias_summary = pd.DataFrame({
    "Method": ["No correction", "Global correction", "Category correction"],
    "Mean absolute residual bias": [mar_none, mar_global, mar_category]
})

print("\nGlobal vs Category-Specific Bias Comparison:")
print(comparison.round(6).to_string())
print("\nResidual gender gap (Woman - Man) after each correction:")
print(residual_df.round(6).to_string())
print("\nMean absolute residual bias:")
print(bias_summary.round(6).to_string(index=False))

#  Merge corrected values back to original (non-exploded) dataframe 
# For incidents with multiple sub-labels, take the mean correction across categories
correction_per_row = (
    df_verbal.groupby("id")["_cat_corr"].mean().rename("_mean_correction")
)
df_out = df.join(correction_per_row, on="id")
df_out["_mean_correction"] = df_out["_mean_correction"].fillna(0.0)
df_out["corrected_gendered_intensity"] = df_out["gendered_intensity"] - df_out["_mean_correction"]
df_out.drop(columns=["_types_list", "_mean_correction"], errors="ignore", inplace=True)

# Save all outputs
df_out.to_csv("bias_corrected_dataset.csv", index=False)
comparison.to_csv("bias_comparison_table.csv")
residual_df.to_csv("residual_bias_table.csv")
bias_summary.to_csv("bias_summary_table.csv", index=False)

print(f"\nSaved corrected dataset to: {"bias_corrected_dataset.csv"}")
print(f"Rows: {len(df_out):,}  |  New column: corrected_gendered_intensity")


## 8. Exploratory Data Analysis


In [ ]:
# packages
import warnings, sys
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy import stats


# load dataset
df = pd.read_csv("bias_corrected_dataset.csv")
log(f"Raw dataset: {df.shape[0]} rows × {df.shape[1]} columns")


# Dataset Overview
section("Dataset Overview")

log(f"\nIncidents total:   {len(df)}")
log(f"Countries:         {df['country'].nunique()} unique")
log(f"Year range:        {int(df['year'].min())} – {int(df['year'].max())}")
log(f"Gender balance:    {df['primary_gender'].value_counts().to_dict()}")

log("\nIncidents per country")
country_counts = df["country"].value_counts()
log(f"Largest:   {country_counts.index[0]} ({country_counts.iloc[0]} incidents)")
log(f"Smallest:  {country_counts.index[-1]} ({country_counts.iloc[-1]} incident(s))")
log(f"\nFull country breakdown:")
print(country_counts.to_string())

log("\nIncidents per year")
print(df["year"].value_counts().sort_index().to_string())

log("\nMissingness")
core_vars = [
    "corrected_gendered_intensity", "primary_gender",
    "country", "year", "backslide_score", "v2lgfemleg",
    "v2clacjstw", "v2csgender", "v2mefemjrn", "v2cldiscw",
    "v2pepwrgen", "v2cldmovew",
]
miss = df[core_vars].isnull().sum()
miss_pct = (miss / len(df) * 100).round(1)
miss_df = pd.DataFrame({"Missing": miss, "Pct": miss_pct})
print(miss_df[miss_df["Missing"] > 0].to_string())

# sample after dropping missing core vars
df_work = df.dropna(subset=core_vars).copy()
df_work["year"] = pd.to_numeric(df_work["year"], errors="coerce")
df_work = df_work[df_work["year"] <= 2024]   # cap at 2024 (2025/26 only partial data)
log(f"\nWorking sample after dropna + capping year at 2024: {len(df_work)} rows")
log(f"  Countries retained: {df_work['country'].nunique()}")
log(f"  Year range:         {int(df_work['year'].min())} – {int(df_work['year'].max())}")


# DV Distribution - corrected_gendered_intensity
section("DV: corrected_gendered_intensity")

dv = df_work["corrected_gendered_intensity"]

log("\nDescriptive statistics")
print(dv.describe().round(6))

# concentration near zero
for tol in [0.001, 0.005, 0.01]:
    n = (dv.abs() <= tol).sum()
    log(f"  Within ±{tol}: {n} rows ({n/len(dv)*100:.1f}%)")

log(f"\n  Skewness:  {dv.skew():.4f}")
log(f"  Kurtosis:  {dv.kurt():.4f}  (normal = 0; >3 = heavy tails)")

# shapiro-wilk on sample
sample = dv.sample(min(2000, len(dv)), random_state=42)
stat, p = stats.shapiro(sample)
log(f"\n  Shapiro-Wilk normality test (n={len(sample)}): W={stat:.4f}, p={p:.4e}")
log(f"  → {'NOT normal' if p < 0.05 else 'Cannot reject normality'}")

log(f"\n  Percentile breakdown of |corrected_gendered_intensity|:")
for q in [0.50, 0.75, 0.90, 0.95, 0.99]:
    val = dv.abs().quantile(q)
    n_above = (dv.abs() > val).sum()
    log(f"  {int(q*100)}th pct: {val:.6f}  ({n_above} incidents above)")


# Main IV — backslide_score
section("Main IV: backslide_score")

bs = df_work["backslide_score"]
log("\nDescriptive statistics")
print(bs.describe().round(4))

n_zero = (bs == 0).sum()
log(f"\n  Incidents with backslide_score = 0:  {n_zero} ({n_zero/len(bs)*100:.1f}%)")
log(f"  Skewness:  {bs.skew():.4f}  (positive = right-skewed)")

log(f"\n  Backslide score by country (mean across incidents):")
bs_by_country = df_work.groupby("country")["backslide_score"].mean().sort_values(ascending=False)
print(bs_by_country.round(3).to_string())


# Key Predictors
section("Key Predictors")

log("\nprimary_gender")
print(df_work["primary_gender"].value_counts())

log("\nyear")
yr_counts = df_work["year"].value_counts().sort_index()
print(yr_counts.to_string())

log("\nStructural gender V-Dem variables")
gender_vars = ["v2clacjstw","v2csgender","v2mefemjrn","v2cldiscw",
               "v2pepwrgen","v2lgfemleg","v2cldmovew"]
print(df_work[gender_vars].describe().round(3))

log("\nCorrelation matrix: gender V-Dem variables")
corr = df_work[gender_vars].corr().round(2)
print(corr)

# flag high-correlation pairs
high = [(a, b, corr.loc[a,b])
        for i,a in enumerate(gender_vars)
        for b in gender_vars[i+1:]
        if abs(corr.loc[a,b]) > 0.7]
log(f"\n  Pairs with |r| > 0.7 (collinear --> cannot use together in one model):")
for a, b, r in high:
    log(f"    {a} ↔ {b}  r = {r:+.2f}")


# Bivariate Relationships with DV
section("Bivariate Relationships With DV")

log("\nbackslide_score vs corrected_gendered_intensity")
r, p = stats.pearsonr(df_work["backslide_score"], df_work["corrected_gendered_intensity"])
r_sp, p_sp = stats.spearmanr(df_work["backslide_score"], df_work["corrected_gendered_intensity"])
log(f"  Pearson r  = {r:.4f}  (p = {p:.4f})")
log(f"  Spearman r = {r_sp:.4f}  (p = {p_sp:.4f})")

log("\nprimary_gender vs corrected_gendered_intensity (mean by group)")
gender_means = df_work.groupby("primary_gender")["corrected_gendered_intensity"].agg(
    ["mean","std","count"]
).round(6)
print(gender_means)
t, p_t = stats.ttest_ind(
    df_work[df_work["primary_gender"]=="Man"]["corrected_gendered_intensity"],
    df_work[df_work["primary_gender"]=="Woman"]["corrected_gendered_intensity"],
)
log(f"  Independent t-test: t = {t:.4f}, p = {p_t:.4f}")
log(f"  --> {'Significant' if p_t < 0.05 else 'Not significant'} mean difference by gender.")

log("\nyear vs corrected_gendered_intensity")
r_yr, p_yr = stats.pearsonr(df_work["year"], df_work["corrected_gendered_intensity"])
log(f"  Pearson r = {r_yr:.4f}  (p = {p_yr:.4f})")

log("\ngender V-Dem variables vs corrected_gendered_intensity")
log("  (Spearman - avoids normality assumption)")
for gv in gender_vars:
    r_s, p_s = stats.spearmanr(df_work[gv], df_work["corrected_gendered_intensity"])
    sig = "**" if p_s < 0.01 else ("*" if p_s < 0.05 else "  ")
    log(f"  {gv:14s}  r = {r_s:+.4f}  p = {p_s:.4f} {sig}")

log("\nMean DV by country (top and bottom 10)")
country_dv = df_work.groupby("country")["corrected_gendered_intensity"].agg(
    ["mean","std","count"]
).round(6).sort_values("mean", ascending=False)
log("  Highest mean gendered intensity (most gendered):")
print(country_dv.head(10).to_string())
log("\n  Lowest mean gendered intensity (least gendered / most negative):")
print(country_dv.tail(10).to_string())


log("EDA complete.")

## 9. Multilevel Regression


In [ ]:
# packages
import warnings, sys
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import arviz as az

NEEDED_COLS = [
    "corrected_gendered_intensity", "primary_gender",
    "country", "year", "backslide_score", "v2lgfemleg",
]


# load + prep data
df = pd.read_csv("bias_corrected_dataset.csv")
df_model = df.dropna(subset=NEEDED_COLS).copy()
df_model["year"] = pd.to_numeric(df_model["year"], errors="coerce")
df_model = df_model[(df_model["year"] >= 2014) & (df_model["year"] <= 2024)].copy()
df_model["year"] = df_model["year"].astype(int)

log(f"Working sample: {len(df_model)} rows, {df_model['country'].nunique()} countries")
log(f"Year range:     {df_model['year'].min()} – {df_model['year'].max()}")

# Z-score numeric predictors
bs_mean, bs_std  = df_model["backslide_score"].mean(), df_model["backslide_score"].std()
lg_mean, lg_std  = df_model["v2lgfemleg"].mean(),      df_model["v2lgfemleg"].std()
yr_mean, yr_std  = df_model["year"].mean(),             df_model["year"].std()

df_model["backslide_z"]  = (df_model["backslide_score"] - bs_mean) / bs_std
df_model["v2lgfemleg_z"] = (df_model["v2lgfemleg"]      - lg_mean) / lg_std
df_model["year_z"]       = (df_model["year"]             - yr_mean) / yr_std

log(f"\nZ-scored: backslide_z (mean={bs_mean:.3f}, SD={bs_std:.3f})")
log(f"          v2lgfemleg_z (mean={lg_mean:.3f}, SD={lg_std:.3f})")
log(f"          year_z       (mean={yr_mean:.3f}, SD={yr_std:.3f})")


# MLE Multilevel Regression
section("MLE Multilevel Regression (statsmodels)")

log("""
Approach: linear mixed-effects model estimated by maximum likelihood
  Fixed effects:  backslide_z, primary_gender, interaction, v2lgfemleg_z, year_z
  Random effect:  random intercept per country (accounts for clustering)
  Estimation:     L-BFGS with up to 2000 iterations
""")

formula = (
    "corrected_gendered_intensity ~ "
    "backslide_z "
    "+ C(primary_gender) "
    "+ C(primary_gender):backslide_z "
    "+ v2lgfemleg_z "
    "+ v2lgfemleg_z:backslide_z "
    "+ year_z"
)

mle_model = smf.mixedlm(
    formula, data=df_model, groups=df_model["country"]
)
mle_result = mle_model.fit(method="lbfgs", maxiter=2000)

log(f"Converged (optimiser):  {mle_result.converged}")
log(f"Log-likelihood:         {mle_result.llf}")

# ICC
re_var  = float(mle_result.cov_re.values[0][0])
res_var = float(mle_result.scale)
total   = re_var + res_var
icc     = re_var / total if total > 0 else 0.0

log(f"\nRandom intercept variance (between countries): {re_var:.10f}")
log(f"Residual variance (within countries):          {res_var:.8f}")
log(f"ICC (intraclass correlation):                  {icc:.6f}")

log("\nFixed effects (key terms)")
params  = mle_result.params
pvalues = mle_result.pvalues
ci      = mle_result.conf_int()
ci.columns = ["hdi94_lb", "hdi94_ub"]

summary_mle = pd.DataFrame({
    "coef":    params.round(8),
    "hdi94_lb":  ci["hdi94_lb"].round(8),
    "hdi94_ub": ci["hdi94_ub"].round(8),
    "p_value": pvalues.round(4),
})
summary_mle = summary_mle[~summary_mle.index.str.contains("Group Var|Intercept", na=False)]
print(summary_mle.to_string())


# Bayesian Multilevel Regression (bambi)
section("Bayesian Multilevel Regression (bambi / PyMC)")

log("""
  Convergence diagnostic: R-hat (Gelman-Rubin statistic)
    R-hat < 1.05 = good convergence
    R-hat > 1.1  = chains did not mix --> estimates unreliable
""")

try:
    import bambi as bmb

    bayes_model = bmb.Model(
        "corrected_gendered_intensity ~ "
        "backslide_z "
        "+ primary_gender "
        "+ primary_gender:backslide_z "
        "+ v2lgfemleg_z "
        "+ v2lgfemleg_z:backslide_z "
        "+ year_z "
        "+ (1|country)",
        data=df_model,
        priors={
            "Intercept|country": {
                "sigma": bmb.Prior("HalfNormal", sigma=0.05)
            }
        },
    )

    log("Sampling (4 chains × 1000 draws + 1000 tune)...")
    idata = bayes_model.fit(
        draws=1000, tune=1000,
        target_accept=0.9,
        random_seed=42,
        progressbar=True,
    )

    # Key fixed effects
    log("\nPosterior summary - fixed effects (94% HDI)")
    key_vars = [
        "Intercept",
        "backslide_z",
        "primary_gender",
        "primary_gender:backslide_z",
        "v2lgfemleg_z",
        "v2lgfemleg_z:backslide_z",
        "year_z",
    ]
    existing = [v for v in key_vars if v in idata.posterior.data_vars]
 
    fixed_summary = az.summary(
        idata, var_names=existing, ci_prob=0.94,
    ).round(6)
    ci_cols = [c for c in fixed_summary.columns
               if any(x in c for x in ["lb", "ub", "hdi", "eti", "ci"])]
    show = ["mean", "sd"] + ci_cols + ["r_hat"]
    show = [c for c in show if c in fixed_summary.columns]
    log(f"  (CI columns: {ci_cols})")
    print(fixed_summary[show])
 
    # Random effect variance
    log("\nPosterior of random intercept SD (1|country_sigma)")
    re_vars = [v for v in idata.posterior.data_vars
               if "sigma" in v.lower() and "country" in v.lower()]
    if not re_vars:
        re_vars = ["1|country_sigma"] if "1|country_sigma" in idata.posterior.data_vars else []
    if re_vars:
        re_summary = az.summary(idata, var_names=re_vars, ci_prob=0.94).round(8)
        re_ci = [c for c in re_summary.columns
                 if any(x in c for x in ["lb", "ub", "hdi", "eti", "ci"])]
        re_show = ["mean", "sd"] + re_ci + ["r_hat"]
        re_show = [c for c in re_show if c in re_summary.columns]
        print(re_summary[re_show])
    else:
        log("  Available posterior variables:")
        for v in sorted(idata.posterior.data_vars):
            log(f"    {v}")
 
    # R-hat check
    log("\nConvergence (R-hat)")
    all_rhat = pd.to_numeric(az.summary(idata, ci_prob=0.94)["r_hat"], errors="coerce").dropna()
    max_rhat = all_rhat.max()
    n_bad    = (all_rhat > 1.05).sum()
    log(f"  Max R-hat across all parameters: {max_rhat:.4f}")
    log(f"  Parameters with R-hat > 1.05:    {n_bad}")
    log(f"  Convergence: {'GOOD' if max_rhat < 1.05 else 'CHECK - some chains may not have mixed'}")

 
except ImportError:
    log("  bambi not available. Install with: pip install bambi")
except Exception as e:
    log(f"  Bayesian model failed: {e}")


log("Step 2 complete.")

## 10. Classification Reframe

In [ ]:
# packages
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, precision_recall_curve,
)
from sklearn.model_selection import train_test_split, StratifiedKFold

 
THRESHOLD_Q = 0.90   # top 10% by |corrected_gendered_intensity|
SEED        = 42
 
NEEDED_COLS = [
    "corrected_gendered_intensity", "primary_gender",
    "country", "year", "backslide_score",
    "v2lgfemleg", "v2csgender", "v2mefemjrn",
    "v2clacjstw", "v2cldiscw", "v2pepwrgen", "v2cldmovew",
]
 
# Gender V-Dem vars significant in EDA bivariate analysis
SIGNIFICANT_GENDER_VARS = ["v2lgfemleg", "v2csgender", "v2mefemjrn"]
# All gender vars for Random Forest
ALL_GENDER_VARS = [
    "v2lgfemleg", "v2csgender", "v2mefemjrn",
    "v2clacjstw", "v2cldiscw", "v2pepwrgen", "v2cldmovew",
]
 
def calibrate_threshold(y_true, y_proba):
    prec, rec, thr = precision_recall_curve(y_true, y_proba)
    f1s = 2 * prec * rec / (prec + rec + 1e-10)
    best = np.argmax(f1s[:-1])
    return float(thr[best])
 
 
def evaluate(name, y_true, y_proba, thr):
    pred = (y_proba > thr).astype(int)
    return {
        "Model":     name,
        "AUC":       round(roc_auc_score(y_true, y_proba), 3),
        "Threshold": round(thr, 3),
        "F1":        round(f1_score(y_true, pred, zero_division=0), 3),
        "Precision": round(precision_score(y_true, pred, zero_division=0), 3),
        "Recall":    round(recall_score(y_true, pred), 3),
    }
 
 
# Target Creation
section("Target Creation")
 
df = pd.read_csv("bias_corrected_dataset.csv")
df_model = df.dropna(subset=NEEDED_COLS).copy()
df_model["year"] = pd.to_numeric(df_model["year"], errors="coerce")
df_model = df_model[
    (df_model["year"] >= 2014) & (df_model["year"] <= 2024)
].copy()
df_model["year"] = df_model["year"].astype(int)
 
threshold = df_model["corrected_gendered_intensity"].abs().quantile(THRESHOLD_Q)
df_model["gendered_flag"] = (
    df_model["corrected_gendered_intensity"].abs() > threshold
).astype(int)
 
log(f"Working sample:  {len(df_model)} rows, {df_model['country'].nunique()} countries")
log(f"Threshold:       {threshold:.6f} (90th percentile of |DV|)")
log(f"Class balance:   {df_model['gendered_flag'].value_counts().to_dict()}")
log(f"Positive rate:   {df_model['gendered_flag'].mean()*100:.1f}%")
 
# Z-scores
bs_mean, bs_std = df_model["backslide_score"].mean(), df_model["backslide_score"].std()
lg_mean, lg_std = df_model["v2lgfemleg"].mean(),      df_model["v2lgfemleg"].std()
yr_mean, yr_std = df_model["year"].mean(),             df_model["year"].std()
 
df_model["backslide_z"]  = (df_model["backslide_score"] - bs_mean) / bs_std
df_model["v2lgfemleg_z"] = (df_model["v2lgfemleg"]      - lg_mean) / lg_std
df_model["v2csgender_z"] = (df_model["v2csgender"]      - df_model["v2csgender"].mean()) / df_model["v2csgender"].std()
df_model["v2mefemjrn_z"] = (df_model["v2mefemjrn"]      - df_model["v2mefemjrn"].mean()) / df_model["v2mefemjrn"].std()
df_model["year_z"]       = (df_model["year"]             - yr_mean) / yr_std
 
# train/test split (stratified to maintain class balance)
train_data, test_data = train_test_split(
    df_model, test_size=0.2, random_state=SEED,
    stratify=df_model["gendered_flag"],
)
 
# drop countries with no positives in training
# (perfect separation, logistic GLM cannot estimate their coefficient)
pos_per_country = train_data.groupby("country")["gendered_flag"].sum()
keep = pos_per_country[pos_per_country > 0].index
dropped = sorted(set(train_data["country"]) - set(keep))
train_data = train_data[train_data["country"].isin(keep)].copy()
test_data  = test_data[test_data["country"].isin(keep)].copy()
 
log(f"\nDropped {len(dropped)} countries with no positives in training:")
log(f"  {dropped}")
log(f"Retained: {len(train_data)} train / {len(test_data)} test")
 
ref_country = train_data["country"].value_counts().idxmax()
log(f"Reference country for scenario predictions: {ref_country}")
 
 
# Gender Variable Comparison
section("Gender Variable Comparison")
 
# baseline: no structural gender variable
base_formula = (
    "gendered_flag ~ C(primary_gender) + backslide_z + year_z + C(country)"
)
base_m    = smf.glm(base_formula, data=train_data, family=sm.families.Binomial()).fit()
base_pr   = base_m.predict(test_data)
base_auc  = roc_auc_score(test_data["gendered_flag"], base_pr)
base_thr  = calibrate_threshold(test_data["gendered_flag"], base_pr)
 
gender_rows = [evaluate("Baseline (no gender var)", test_data["gendered_flag"], base_pr, base_thr)]
 
gv_z_map = {
    "v2lgfemleg": "v2lgfemleg_z",
    "v2csgender": "v2csgender_z",
    "v2mefemjrn": "v2mefemjrn_z",
}
gv_results = {}
 
for gv in SIGNIFICANT_GENDER_VARS:
    gvz = gv_z_map[gv]
    formula = (
        f"gendered_flag ~ C(primary_gender) + backslide_z "
        f"+ {gvz} + year_z + C(country)"
    )
    try:
        m   = smf.glm(formula, data=train_data, family=sm.families.Binomial()).fit()
        pr  = m.predict(test_data)
        auc = roc_auc_score(test_data["gendered_flag"], pr)
        thr = calibrate_threshold(test_data["gendered_flag"], pr)
        OR  = round(np.exp(m.params[gvz]), 3)
        p   = round(m.pvalues[gvz], 4)
        ci_low  = round(np.exp(m.conf_int().loc[gvz, 0]), 3)
        ci_high = round(np.exp(m.conf_int().loc[gvz, 1]), 3)
        gv_results[gv] = {"model": m, "proba": pr, "OR": OR, "p": p}
        row = evaluate(gv, test_data["gendered_flag"], pr, thr)
        row["OR"] = OR; row["OR_CI"] = f"[{ci_low}, {ci_high}]"; row["p"] = p
        gender_rows.append(row)
        log(f"  {gv:14s}  AUC={auc:.3f}  OR={OR:.3f} {row['OR_CI']}  p={p:.4f}")
    except Exception as e:
        log(f"  {gv}: failed ({e})")
 
gender_comp = pd.DataFrame(gender_rows).sort_values("AUC", ascending=False)
log("\nGender variable comparison — ranked by AUC:")
cols_to_show = [c for c in ["Model","AUC","F1","OR","OR_CI","p"] if c in gender_comp.columns]
print(gender_comp[cols_to_show].to_string(index=False))
 
# select best variable for next step
best_gv = max(gv_results, key=lambda g: roc_auc_score(
    test_data["gendered_flag"], gv_results[g]["proba"]
))
best_gvz = gv_z_map[best_gv]
log(f"\nSelected for theory-testing block: {best_gv} ({best_gvz})")
 
 
# Theory-Testing Models A → B → C
section("Theory-Testing Logistic Models A → B → C")
 
formulas = {
    "A — main effects": (
        f"gendered_flag ~ C(primary_gender) + backslide_z "
        f"+ {best_gvz} + year_z + C(country)"
    ),
    "B — + gender×backslide": (
        f"gendered_flag ~ C(primary_gender) + backslide_z "
        f"+ {best_gvz} + year_z "
        f"+ C(primary_gender):backslide_z "
        f"+ C(country)"
    ),
    "C — + both moderators": (
        f"gendered_flag ~ C(primary_gender) + backslide_z "
        f"+ {best_gvz} + year_z "
        f"+ C(primary_gender):backslide_z "
        f"+ {best_gvz}:backslide_z "
        f"+ C(country)"
    ),
}
 
fitted = {}
theory_rows = []
 
for name, formula in formulas.items():
    log(f"Fitting Model {name[0]}...")
    try:
        m   = smf.glm(formula, data=train_data, family=sm.families.Binomial()).fit()
        pr  = m.predict(test_data)
        thr = calibrate_threshold(test_data["gendered_flag"], pr)
        fitted[name] = {"model": m, "proba": pr, "threshold": thr}
        theory_rows.append(evaluate(name, test_data["gendered_flag"], pr, thr))
        log(f"  AUC={theory_rows[-1]['AUC']:.3f}  F1={theory_rows[-1]['F1']:.3f}  "
            f"Recall={theory_rows[-1]['Recall']:.3f}")
    except Exception as e:
        log(f"  ⚠ {e}")
 
log("\nModel comparison (calibrated threshold):")
print(pd.DataFrame(theory_rows).to_string(index=False))
 
 
# Random Forest
section("Random Forest")
 
 
# feature matrix
le = LabelEncoder()
gender_enc_train = le.fit_transform(train_data["primary_gender"].astype(str))
gender_enc_test  = le.transform(test_data["primary_gender"].astype(str))
 
country_dummies_train = pd.get_dummies(train_data["country"], prefix="c")
country_dummies_test  = pd.get_dummies(test_data["country"],  prefix="c")
country_dummies_test  = country_dummies_test.reindex(
    columns=country_dummies_train.columns, fill_value=0
)
 
# Z-scores for all gender vars
for gv in ALL_GENDER_VARS:
    m_g = df_model[gv].mean()
    s_g = df_model[gv].std()
    df_model[f"{gv}_z"] = (df_model[gv] - m_g) / s_g
 
gv_z_cols = [f"{gv}_z" for gv in ALL_GENDER_VARS]
for c in gv_z_cols:
    if c not in train_data.columns:
        m_g = df_model[c.replace("_z","")].mean()
        s_g = df_model[c.replace("_z","")].std()
        train_data[c] = (train_data[c.replace("_z","")] - m_g) / s_g
        test_data[c]  = (test_data[c.replace("_z","")]  - m_g) / s_g
 
rf_num_cols = ["backslide_z", "year_z"] + gv_z_cols
 
X_train = pd.concat([
    train_data[rf_num_cols].reset_index(drop=True),
    pd.Series(gender_enc_train, name="gender_enc"),
    country_dummies_train.reset_index(drop=True),
], axis=1)
 
X_test = pd.concat([
    test_data[rf_num_cols].reset_index(drop=True),
    pd.Series(gender_enc_test, name="gender_enc"),
    country_dummies_test.reset_index(drop=True),
], axis=1)
 
y_train = train_data["gendered_flag"].reset_index(drop=True)
y_test  = test_data["gendered_flag"].reset_index(drop=True)
 
log("Fitting Random Forest (500 trees, balanced class weight)...")
rf = RandomForestClassifier(
    n_estimators=500, class_weight="balanced",
    min_samples_leaf=5, random_state=SEED, n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_thr   = calibrate_threshold(y_test, rf_proba)
rf_row   = evaluate("Random Forest", y_test, rf_proba, rf_thr)
log(f"  AUC={rf_row['AUC']:.3f}  F1={rf_row['F1']:.3f}  Recall={rf_row['Recall']:.3f}")
 
log("\nFeature importance - gender V-Dem variables:")
fi = pd.DataFrame({
    "feature":    X_train.columns,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)
 
non_country = fi[~fi["feature"].str.startswith("c_")].copy()
gender_fi   = non_country[non_country["feature"].isin(gv_z_cols)]
other_fi    = non_country[~non_country["feature"].isin(gv_z_cols)]
print(gender_fi.round(4).to_string(index=False))
log("\nNon-gender predictors (context):")
print(other_fi.round(4).to_string(index=False))


# 5-fold Stratified Cross-Validation
section("5-fold Stratified Cross-Validation")

def build_dataset(percentile):
    df_out = df_work.copy()
    thr = df_out["corrected_gendered_intensity"].abs().quantile(percentile)
    df_out["gendered_flag"] = (
        df_out["corrected_gendered_intensity"].abs() > thr
    ).astype(int)
    return df_out, thr
 
def split_and_filter(df_in, seed=SEED):
    train, test = train_test_split(
        df_in, test_size=0.2, random_state=seed,
        stratify=df_in["gendered_flag"],
    )
    pos = train.groupby("country")["gendered_flag"].sum()
    keep = pos[pos > 0].index
    return (train[train["country"].isin(keep)].copy(),
            test[test["country"].isin(keep)].copy())

df_cv, _ = build_dataset(0.90)
df_cv["backslide_z"]  = (df_cv["backslide_score"] - df_cv["backslide_score"].mean()) / df_cv["backslide_score"].std()
df_cv["year_z"]       = (df_cv["year"]             - df_cv["year"].mean())             / df_cv["year"].std()
df_cv["v2csgender_z"] = (df_cv["v2csgender"]       - df_cv["v2csgender"].mean())       / df_cv["v2csgender"].std()
for gv in ALL_GENDER_VARS:
    df_cv[f"{gv}_z"] = (df_cv[gv] - df_cv[gv].mean()) / df_cv[gv].std()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for gv in ALL_GENDER_VARS:
    m_g = df_cv[gv].mean()
    s_g = df_cv[gv].std()
    df_cv[f"{gv}_z"] = (df_cv[gv] - m_g) / s_g

bs_mean, bs_std = df_work["backslide_score"].mean(), df_work["backslide_score"].std()
yr_mean, yr_std = df_work["year"].mean(),            df_work["year"].std()
cs_mean, cs_std = df_work["v2csgender"].mean(),      df_work["v2csgender"].std()

df_work["backslide_z"]  = (df_work["backslide_score"] - bs_mean) / bs_std
df_work["year_z"]       = (df_work["year"]             - yr_mean) / yr_std
df_work["v2csgender_z"] = (df_work["v2csgender"]       - cs_mean) / cs_std

MODEL_C_FORMULA = (
    "gendered_flag ~ C(primary_gender) + backslide_z "
    "+ v2csgender_z + year_z "
    "+ C(primary_gender):backslide_z "
    "+ v2csgender_z:backslide_z "
    "+ C(country)"
)
 
logistic_aucs = []
rf_aucs       = []
le_cv = LabelEncoder()
le_cv.fit(df_cv["primary_gender"].astype(str))
 
RF_NUM_COLS = ["backslide_z", "year_z"] + [f"{g}_z" for g in ALL_GENDER_VARS]
 
for fold, (train_idx, val_idx) in enumerate(
    skf.split(df_cv, df_cv["gendered_flag"])
):
    train_f = df_cv.iloc[train_idx].copy()
    val_f   = df_cv.iloc[val_idx].copy()
 
    # drop countries with no positives in this fold
    pos_f = train_f.groupby("country")["gendered_flag"].sum()
    keep_f = pos_f[pos_f > 0].index
    train_f = train_f[train_f["country"].isin(keep_f)].copy()
    val_f   = val_f[val_f["country"].isin(keep_f)].copy()
 
    if val_f["gendered_flag"].nunique() < 2:
        log(f"  Fold {fold+1}: skipped (no positives in val fold after filter)")
        continue
 
    # logistic model C
    try:
        m_cv   = smf.glm(MODEL_C_FORMULA, data=train_f,
                         family=sm.families.Binomial()).fit()
        pr_cv  = m_cv.predict(val_f)
        auc_cv = roc_auc_score(val_f["gendered_flag"], pr_cv)
        logistic_aucs.append(auc_cv)
        log(f"  Fold {fold+1} - Logistic AUC: {auc_cv:.3f}", )
    except Exception as e:
        log(f"  Fold {fold+1} - Logistic failed: {e}")
 
    # Random Forest
    try:
        c_dum_tr = pd.get_dummies(train_f["country"], prefix="c")
        c_dum_vl = pd.get_dummies(val_f["country"],   prefix="c")
        c_dum_vl = c_dum_vl.reindex(columns=c_dum_tr.columns, fill_value=0)
 
        X_tr = pd.concat([
            train_f[RF_NUM_COLS].reset_index(drop=True),
            pd.Series(le_cv.transform(train_f["primary_gender"].astype(str)),
                      name="gender_enc"),
            c_dum_tr.reset_index(drop=True),
        ], axis=1)
        X_vl = pd.concat([
            val_f[RF_NUM_COLS].reset_index(drop=True),
            pd.Series(le_cv.transform(val_f["primary_gender"].astype(str)),
                      name="gender_enc"),
            c_dum_vl.reset_index(drop=True),
        ], axis=1)
        y_tr = train_f["gendered_flag"].reset_index(drop=True)
        y_vl = val_f["gendered_flag"].reset_index(drop=True)
 
        rf_cv = RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            min_samples_leaf=5, random_state=SEED, n_jobs=-1,
        )
        rf_cv.fit(X_tr, y_tr)
        auc_rf = roc_auc_score(y_vl, rf_cv.predict_proba(X_vl)[:, 1])
        rf_aucs.append(auc_rf)
        log(f"           - RF AUC:      {auc_rf:.3f}")
    except Exception as e:
        log(f"  Fold {fold+1} i RF failed: {e}")
 
log(f"\nCross-validation results (5-fold stratified):")
if logistic_aucs:
    log(f"  Logistic Model C:  mean AUC = {np.mean(logistic_aucs):.3f} "
        f"± {np.std(logistic_aucs):.3f}  "
        f"(range {min(logistic_aucs):.3f}–{max(logistic_aucs):.3f})")
if rf_aucs:
    log(f"  Random Forest:     mean AUC = {np.mean(rf_aucs):.3f} "
        f"± {np.std(rf_aucs):.3f}  "
        f"(range {min(rf_aucs):.3f}–{max(rf_aucs):.3f})")
 
 
# Full Model Comparison
section("Full Model Comparison")
 
all_rows = theory_rows + [rf_row]
comparison = pd.DataFrame(all_rows)
print(comparison[["Model","AUC","Threshold","F1","Precision","Recall"]].to_string(index=False))
 
# Confusion matrix for Model C
log("\nConfusion matrix - Model C:")
best_name = "C — + both moderators"
if best_name in fitted:
    pr_c  = fitted[best_name]["proba"]
    thr_c = fitted[best_name]["threshold"]
    pred_c = (pr_c > thr_c).astype(int)
    cm = confusion_matrix(test_data["gendered_flag"], pred_c)
    log(f"  True negatives  (correctly predicted non-gendered): {cm[0,0]}")
    log(f"  False positives (non-gendered predicted gendered):  {cm[0,1]}")
    log(f"  False negatives (gendered missed):                  {cm[1,0]}")
    log(f"  True positives  (correctly predicted gendered):     {cm[1,1]}")
 
 
# Odds Ratios for Model C
section("Odds Ratios for Model C")
 
log("Reading guide:")
log("  OR > 1 = predictor raises odds of being a highly-gendered incident")
log("  OR < 1 = predictor lowers odds")
log("  p < 0.05 = statistically significant at conventional threshold")
 
if best_name in fitted:
    m_c     = fitted[best_name]["model"]
    params  = m_c.params
    pvalues = m_c.pvalues
    ci      = m_c.conf_int()
    ci.columns = ["CI_low", "CI_high"]
 
    or_table = pd.DataFrame({
        "OR":      np.exp(params),
        "CI_low":  np.exp(ci["CI_low"]),
        "CI_high": np.exp(ci["CI_high"]),
        "p_value": pvalues,
    }).round(4)
 
    key = [i for i in or_table.index
           if "C(country)" not in i and "Intercept" not in i]
    print(or_table.loc[key].to_string())
 
 
# Scenario Predictions
section("Scenario Predictions (new data)")
 
bs_mean_val = df_model["backslide_score"].mean()
lg_mean_val = df_model[best_gv].mean()
 

def make_row(gender, year_val, bs_raw, gv_raw):
    return pd.DataFrame([{
        "primary_gender":  gender,
        "country":         ref_country,
        "year":            year_val,
        "backslide_z":     (bs_raw - bs_mean) / bs_std,
        best_gvz:          (gv_raw - lg_mean) / lg_std,
        "year_z":          (year_val - yr_mean) / yr_std,
        "gendered_flag":   0,
    }])
 
 
def predict_logistic(row_df):
    if best_name not in fitted:
        return np.nan
    return round(float(fitted[best_name]["model"].predict(row_df).iloc[0]) * 100, 1)
 
 
def predict_rf(gender, year_val, bs_raw, gv_raw):
    bs_z  = (bs_raw - bs_mean) / bs_std
    gv_z  = (gv_raw - lg_mean) / lg_std
    yr_z  = (year_val - yr_mean) / yr_std
    g_enc = int(le.transform([gender])[0])
    row   = pd.DataFrame([{
        "backslide_z": bs_z, "year_z": yr_z, "gender_enc": g_enc,
    }])
    # add all gender var z cols
    for gvc in gv_z_cols:
        if gvc == best_gvz:
            row[gvc] = gv_z
        else:
            row[gvc] = 0.0
    # add country dummies
    for col in country_dummies_train.columns:
        row[col] = 1 if col == f"c_{ref_country}" else 0
    row = row[X_train.columns]
    return round(float(rf.predict_proba(row)[0, 1]) * 100, 1)
 
 
# Scenario 1 - gender comparison
log("Scenario 1: Male vs Female journalist (2020, mean backsliding)")
for gender in ["Man", "Woman"]:
    row = make_row(gender, 2020, bs_mean_val, lg_mean_val)
    lp  = predict_logistic(row)
    rp  = predict_rf(gender, 2020, bs_mean_val, lg_mean_val)
    log(f"  {gender:6s}  →  Logistic: {lp}%   RF: {rp}%")
 
# Scenario 2 - time trend
log("\nScenario 2: Time trend for female journalist (mean backsliding)")
for yr in [2014, 2016, 2018, 2020, 2022, 2024]:
    row = make_row("Woman", yr, bs_mean_val, lg_mean_val)
    lp  = predict_logistic(row)
    rp  = predict_rf("Woman", yr, bs_mean_val, lg_mean_val)
    log(f"  {yr}  →  Logistic: {lp}%   RF: {rp}%")
 
# Scenario 3 - backsliding 
log("\nScenario 3: Backsliding level, female journalist (2020)")
bs_low  = max(0, bs_mean_val - bs_std)
bs_high = bs_mean_val + bs_std
for label, bs_val in [
    ("Low  (-1 SD)", bs_low),
    ("Mean (  0  )", bs_mean_val),
    ("High (+1 SD)", bs_high),
]:
    row = make_row("Woman", 2020, bs_val, lg_mean_val)
    lp  = predict_logistic(row)
    rp  = predict_rf("Woman", 2020, bs_val, lg_mean_val)
    log(f"  {label}  →  Logistic: {lp}%   RF: {rp}%")
 
# Scenario 4 - 2×2 gender × backsliding
log("\nScenario 4: Gender × backsliding 2×2 (2020)")
for gender in ["Man", "Woman"]:
    for label, bs_val in [("low backslide", bs_low), ("high backslide", bs_high)]:
        row = make_row(gender, 2020, bs_val, lg_mean_val)
        lp  = predict_logistic(row)
        rp  = predict_rf(gender, 2020, bs_val, lg_mean_val)
        log(f"  {gender:6s} × {label}  →  Logistic: {lp}%   RF: {rp}%")
 
# Treshold Sensitivity Analysis
section("Treshold Sensitivity Analysis")
 
sensitivity_rows = []
for pct in [0.85, 0.90, 0.95]:
    df_pct, thr_val = build_dataset(pct)
    n_pos = df_pct["gendered_flag"].sum()
    train_p, test_p = split_and_filter(df_pct)
 
    try:
        m = smf.glm(MODEL_C_FORMULA, data=train_p,
                    family=sm.families.Binomial()).fit()
        pr  = m.predict(test_p)
        thr_c = calibrate_threshold(test_p["gendered_flag"], pr)
        auc = roc_auc_score(test_p["gendered_flag"], pr)
 
        import numpy as xnp
        def get_or(term):
            if term in m.params.index:
                return (round(float(xnp.exp(m.params[term])), 3),
                        round(float(m.pvalues[term]), 4))
            return (np.nan, np.nan)
 
        or_gender, p_gender = get_or("C(primary_gender)[T.Woman]")
        or_year,   p_year   = get_or("year_z")
        or_bs,     p_bs     = get_or("backslide_z")
        or_inter,  p_inter  = get_or(
            "C(primary_gender)[T.Woman]:backslide_z"
        )
 
        sensitivity_rows.append({
            "Threshold":    f"{int(pct*100)}th pct",
            "N positives":  n_pos,
            "AUC":          round(auc, 3),
            "OR gender":    or_gender, "p gender":   p_gender,
            "OR year":      or_year,   "p year":     p_year,
            "OR backslide": or_bs,     "p backslide":p_bs,
            "OR interact":  or_inter,  "p interact": p_inter,
        })
        log(f"  {int(pct*100)}th pct | n_pos={n_pos} | AUC={auc:.3f} | "
            f"OR_gender={or_gender}(p={p_gender}) | "
            f"OR_year={or_year}(p={p_year}) | "
            f"OR_interact={or_inter}(p={p_inter})")
    except Exception as e:
        log(f"  {int(pct*100)}th pct failed: {e}")
 
log("\nSensitivity table:")
if sensitivity_rows:
    print(pd.DataFrame(sensitivity_rows).to_string(index=False))
 

log("Step 3 complete.")


## 11. Visualisations

In [ ]:
# Visualisations

#packages
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, precision_recall_curve, roc_curve,
)

COLORS = {
    "A": "#2C7BB6", "B": "#74ADD1", "C": "#D7191C",
    "RF": "#1A9641", "Woman": "#D7191C", "Man": "#2C7BB6",
}

# re-fit model C and RF on the 90th-pct 80/20 split for plot data
df_90, _ = build_dataset(0.90)

for gv in ALL_GENDER_VARS:
    m_g = df_90[gv].mean()
    s_g = df_90[gv].std()
    df_90[f"{gv}_z"] = (df_90[gv] - m_g) / s_g

train_v, test_v = split_and_filter(df_90)
ref_country = train_v["country"].value_counts().idxmax()
 
m_c_v = smf.glm(MODEL_C_FORMULA, data=train_v,
                family=sm.families.Binomial()).fit()
 
# logistic variants for ROC
m_a = smf.glm(
    "gendered_flag ~ C(primary_gender) + backslide_z + v2csgender_z + year_z + C(country)",
    data=train_v, family=sm.families.Binomial()
).fit()
m_b = smf.glm(
    "gendered_flag ~ C(primary_gender) + backslide_z + v2csgender_z + year_z "
    "+ C(primary_gender):backslide_z + C(country)",
    data=train_v, family=sm.families.Binomial()
).fit()
 
pr_a   = m_a.predict(test_v)
pr_b   = m_b.predict(test_v)
pr_c   = m_c_v.predict(test_v)
 
# RF
le_v = LabelEncoder()
c_dum_tr_v = pd.get_dummies(train_v["country"], prefix="c")
c_dum_te_v = pd.get_dummies(test_v["country"],  prefix="c")
c_dum_te_v = c_dum_te_v.reindex(columns=c_dum_tr_v.columns, fill_value=0)
X_tr_v = pd.concat([
    train_v[RF_NUM_COLS].reset_index(drop=True),
    pd.Series(le_v.fit_transform(train_v["primary_gender"].astype(str)),
              name="gender_enc"),
    c_dum_tr_v.reset_index(drop=True),
], axis=1)
X_te_v = pd.concat([
    test_v[RF_NUM_COLS].reset_index(drop=True),
    pd.Series(le_v.transform(test_v["primary_gender"].astype(str)),
              name="gender_enc"),
    c_dum_te_v.reset_index(drop=True),
], axis=1)
y_te_v = test_v["gendered_flag"].reset_index(drop=True)
 
rf_v = RandomForestClassifier(
    n_estimators=500, class_weight="balanced",
    min_samples_leaf=5, random_state=SEED, n_jobs=-1,
)
rf_v.fit(X_tr_v, train_v["gendered_flag"].reset_index(drop=True))
pr_rf_v = rf_v.predict_proba(X_te_v)[:, 1]
 
 
# FIGURE 1 - DV Distribution
dv = df_work["corrected_gendered_intensity"]
thr_90 = dv.abs().quantile(0.90)
 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Distribution of corrected_gendered_intensity", fontsize=13, fontweight="bold")
 
# left - full distribution
axes[0].hist(dv, bins=120, color="#2C7BB6", alpha=0.75, edgecolor="none")
axes[0].axvline(-thr_90, color="#D7191C", lw=1.5, ls="--", label=f"90th pct threshold (±{thr_90:.4f})")
axes[0].axvline( thr_90, color="#D7191C", lw=1.5, ls="--")
axes[0].set_xlabel("corrected_gendered_intensity")
axes[0].set_ylabel("Count")
axes[0].set_title("Full distribution")
axes[0].legend(fontsize=9)
 
# right - clip to ±0.01 to show concentration
dv_clip = dv.clip(-0.01, 0.01)
axes[1].hist(dv_clip, bins=80, color="#2C7BB6", alpha=0.75, edgecolor="none")
axes[1].axvline(-thr_90, color="#D7191C", lw=1.5, ls="--", label=f"±{thr_90:.4f} threshold")
axes[1].axvline( thr_90, color="#D7191C", lw=1.5, ls="--")
axes[1].set_xlabel("corrected_gendered_intensity (clipped to ±0.01)")
axes[1].set_title("Zoomed: clipped to ±0.01")
axes[1].annotate(
    "80.6% of values\nwithin ±0.001 of zero",
    xy=(0, axes[1].get_ylim()[1] * 0.85),
    ha="center", fontsize=9, color="#555",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#ccc"),
)
axes[1].legend(fontsize=9)
 
plt.tight_layout()
plt.savefig("fig1_dv_distribution.png", bbox_inches="tight")
plt.close()
log("  Saved.")
 
 
# FIGURE 2 — ROC Curves
fig, ax = plt.subplots(figsize=(6, 6))
y_true_v = test_v["gendered_flag"].values
 
for name, pr, color in [
    ("Logistic A — main effects",   pr_a,    COLORS["A"]),
    ("Logistic B — +gender×bs",     pr_b,    COLORS["B"]),
    ("Logistic C — both moderators",pr_c,    COLORS["C"]),
    ("Random Forest",               pr_rf_v, COLORS["RF"]),
]:
    fpr, tpr, _ = roc_curve(y_true_v, pr)
    auc = roc_auc_score(y_true_v, pr)
    ax.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC={auc:.3f})")
 
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Random chance (AUC=0.5)")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC Curves - All Models", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.savefig("fig2_roc_curves.png", bbox_inches="tight")
plt.close()
log("  Saved.")
 
 
# FIGURE 3 - Feature Importance
fi_all = pd.DataFrame({
    "feature":    X_tr_v.columns,
    "importance": rf_v.feature_importances_,
}).sort_values("importance", ascending=False)
 
fi_plot = fi_all[~fi_all["feature"].str.startswith("c_")].copy()
fi_plot["feature"] = fi_plot["feature"].str.replace("_z", "").str.replace("_enc", " (gender)")
 
gender_feats = [f.replace("_z","") for f in [f"{g}_z" for g in ALL_GENDER_VARS]]
fi_plot["category"] = fi_plot["feature"].apply(
    lambda x: "Gender V-Dem" if x in gender_feats else "Core predictor"
)
palette = {"Gender V-Dem": "#ABD9E9", "Core predictor": "#D7191C"}
colors_fi = fi_plot["category"].map(palette)
 
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(fi_plot["feature"], fi_plot["importance"],
               color=colors_fi, edgecolor="none", height=0.7)
ax.set_xlabel("Feature Importance (Mean Decrease in Impurity)", fontsize=11)
ax.set_title("Random Forest — Feature Importance\n(country fixed effects excluded)",
             fontsize=12, fontweight="bold")
ax.invert_yaxis()
patches = [
    mpatches.Patch(color=palette["Core predictor"],  label="Core predictor"),
    mpatches.Patch(color=palette["Gender V-Dem"],    label="Gender V-Dem variable"),
]
ax.legend(handles=patches, fontsize=9)
plt.tight_layout()
plt.savefig("fig3_feature_importance.png", bbox_inches="tight")
plt.close()
log("  Saved.")
 
 
# FIGURE 4 — Time Trend (Scenario Predictions)
years       = list(range(2014, 2025))
bs_mean_val = df_work["backslide_score"].mean()
cs_mean_val = df_work["v2csgender"].mean()
 
def logistic_prob(gender, year_val):
    row = pd.DataFrame([{
        "primary_gender":  gender,
        "country":         ref_country,
        "year":            year_val,
        "backslide_z":     (bs_mean_val - bs_mean) / bs_std,
        "v2csgender_z":    (cs_mean_val - cs_mean) / cs_std,
        "year_z":          (year_val    - yr_mean) / yr_std,
        "gendered_flag":   0,
    }])
    return float(m_c_v.predict(row).iloc[0]) * 100
 
 
man_probs   = [logistic_prob("Man",   y) for y in years]
woman_probs = [logistic_prob("Woman", y) for y in years]
 
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(years, man_probs,   "o-", color=COLORS["Man"],   lw=2.5,
        label="Male journalist",   markersize=6)
ax.plot(years, woman_probs, "s-", color=COLORS["Woman"], lw=2.5,
        label="Female journalist", markersize=6)
ax.fill_between(years, man_probs, woman_probs,
                alpha=0.10, color="#555", label="Gender gap")
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Predicted probability of\nhighly-gendered incident (%)", fontsize=11)
ax.set_title(f"Predicted Probability Over Time\n(reference country: {ref_country}, mean backsliding)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45)
plt.tight_layout()
plt.savefig("fig4_time_trend.png", bbox_inches="tight")
plt.close()
log("  Saved.")
 
 
# FIGURE 5 — Gender × Backsliding (2×2)
bs_low  = max(0, bs_mean_val - df_work["backslide_score"].std())
bs_high = bs_mean_val + df_work["backslide_score"].std()

def lp2(gender, year_val, bs_raw, cs_raw):
    row = pd.DataFrame([{
        "primary_gender": gender, "country": ref_country,
        "year":           year_val,
        "backslide_z":    (bs_raw  - bs_mean) / bs_std,
        "v2csgender_z":   (cs_raw  - cs_mean) / cs_std,
        "year_z":         (year_val - yr_mean) / yr_std,
        "gendered_flag":  0,
    }])
    return float(m_c_v.predict(row).iloc[0]) * 100
 
labels = [
    "Man\nLow backslide", "Man\nHigh backslide",
    "Woman\nLow backslide", "Woman\nHigh backslide",
]
values = [
    lp2("Man",   2020, bs_low,  cs_mean_val),
    lp2("Man",   2020, bs_high, cs_mean_val),
    lp2("Woman", 2020, bs_low,  cs_mean_val),
    lp2("Woman", 2020, bs_high, cs_mean_val),
]
bar_colors = [COLORS["Man"], COLORS["Man"], COLORS["Woman"], COLORS["Woman"]]
alphas = [0.55, 1.0, 0.55, 1.0]
 
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=bar_colors, alpha=0.85, edgecolor="white",
              linewidth=1.2, width=0.55)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
 
ax.set_ylabel("Predicted probability of\nhighly-gendered incident (%)", fontsize=11)
ax.set_title("Gender × Backsliding: Predicted Probabilities (2020)\n"
             f"(Logistic Model C, reference country: {ref_country})",
             fontsize=12, fontweight="bold")
patches = [
    mpatches.Patch(color=COLORS["Man"],   label="Male journalist"),
    mpatches.Patch(color=COLORS["Woman"], label="Female journalist"),
]
ax.legend(handles=patches, fontsize=10)
ax.set_ylim(0, max(values) * 1.25)
plt.tight_layout()
plt.savefig("fig5_gender_backslide.png", bbox_inches="tight")
plt.close()
log("  Saved.")

log("All visualisations saved.")
